# 🧠 MedAssist-AI-IMAGE — Deep Learning: From Scratch → Transfer Learning
**Clasificación de medicamentos en 5 macroclases terapéuticas**

---

## Objetivo y estructura

Este notebook implementa un pipeline completo de Deep Learning para clasificar imágenes de medicamentos en las mismas 5 clases del notebook de ML, permitiendo una **comparación directa y justa** entre ambos enfoques.

El pipeline sigue una progresión metodológica deliberada:

1. **CNN From Scratch** — diseñada a mano, entrenada desde cero. Sin conocimiento previo.
2. **CNN From Scratch Mejorada** — mismo diseño con dropout más agresivo, data augmentation extra y cosine annealing.
3. **Transfer Learning Feature Extractor** — backbone MobileNetV2 congelado, solo se entrena la cabeza.
4. **Transfer Learning Fine-Tuning** — MobileNetV2 con backbone descongelado progresivamente.
5. **Transfer Learning ResNet-50 Fine-Tuning** — backbone más potente, comparativa con MobileNetV2.

**¿Por qué esta progresión?** Cada paso añade una decisión de diseño nueva (más datos sintéticos → backbone preentrenado → fine-tuning) cuyo efecto se puede medir comparando con el paso anterior. Es la misma lógica de los ablation studies del notebook de ML.

---

## Estructura del notebook

**BLOQUE 0 — Configuración**
1. Imports, rutas y configuración global (GPU / Lightning AI)
2. Carga del dataset y remapeo de rutas

**BLOQUE 1 — Dataset y DataLoaders**
3. Transforms y clase MedDataset
4. DataLoaders con WeightedRandomSampler
5. Visualización del batch

**BLOQUE 2 — CNN From Scratch**
6. Arquitectura MedCNN (4 bloques conv)
7. Entrenamiento V1 (base)
8. Entrenamiento V2 (mejorada: más dropout + cosine annealing)
9. Comparativa V1 vs V2

**BLOQUE 3 — Transfer Learning**
10. MobileNetV2 — Feature Extractor (backbone congelado)
11. MobileNetV2 — Fine-Tuning progresivo
12. ResNet-50 — Fine-Tuning

**BLOQUE 4 — Evaluación final**
13. Evaluación en test de todos los modelos
14. Matrices de confusión
15. Análisis de errores — imágenes mal clasificadas
16. Grad-CAM — ¿qué zonas activa la red?

**BLOQUE 5 — Comparativa global**
17. DL vs ML clásico
18. Conclusiones


---
# BLOQUE 0 — Configuración

## 1. Imports y configuración global

### Nota sobre GPU
En Lightning AI / Google Colab con GPU T4 el entrenamiento completo tarda ~25-40 min.
En CPU puede tardar 3-5 horas — considera reducir `N_EPOCHS_SCRATCH = 10` y `N_EPOCHS_TL = 8`.


In [ ]:
import os, random, warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from torchvision.models import (
    MobileNet_V2_Weights, ResNet50_Weights
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
import joblib

# ─────────────────────────────────────────────────────────────────────────────
# RUTAS — EDITAR SEGÚN LA MÁQUINA
# ─────────────────────────────────────────────────────────────────────────────
# IMAGES_DIR: carpeta raíz donde están las imágenes EN ESTA MÁQUINA
#   Misma máquina que el EDA  → IMAGES_DIR = None
#   Lightning AI              → IMAGES_DIR = Path('/teamspace/studios/this_studio/images')
#   Google Colab              → IMAGES_DIR = Path('/content/drive/MyDrive/med_images')
IMAGES_DIR   = None

CSV_BALANCED = 'output/dataset_balanced.csv'
CSV_ORIGINAL = 'output/dataset_split.csv'
OUTPUT_DIR   = Path('output_dl')
OUTPUT_DIR.mkdir(exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# HIPERPARÁMETROS GLOBALES
# ─────────────────────────────────────────────────────────────────────────────
SEED             = 42
IMG_SIZE         = 224
BATCH            = 32
N_EPOCHS_SCRATCH = 25     # Scratch — más épocas porque aprende más lento
N_EPOCHS_TL      = 20     # Transfer Learning
PATIENCE         = 7      # Early stopping
LR_SCRATCH       = 1e-3
LR_TL_HEAD       = 3e-4   # lr para la cabeza nueva
LR_TL_BACK       = 3e-5   # lr para el backbone (fine-tuning)
UNFREEZE_EPOCH   = 5      # Época en que se descongela el backbone

CLASES = [
    'Cardiovascular',
    'Neurología y psiquiatría',
    'Antiinfecciosos sistémicos',
    'Respiratorio',
    'otros'
]
N_CLASES = len(CLASES)
PALETTE  = dict(zip(CLASES, ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']))

# Reproducibilidad
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  ⚠ Sin GPU — considera reducir N_EPOCHS_SCRATCH=10, N_EPOCHS_TL=8')

plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11})
sns.set_style('whitegrid')
print(f'\n✔ Configuración lista | IMG_SIZE={IMG_SIZE} | BATCH={BATCH}')


## 2. Carga del dataset y remapeo de rutas

In [ ]:
# ── Remapeo de rutas (necesario si ejecutas en otra máquina) ──────────────────
def remap_image_paths(df, base_dir):
    if base_dir is None:
        n_ok = df['image_path'].head(20).apply(lambda p: Path(p).exists()).sum()
        if n_ok == 0:
            raise RuntimeError(
                'Ninguna ruta de imagen válida y IMAGES_DIR=None.\n'
                f'  Ejemplo ruta CSV: {df["image_path"].iloc[0]}\n'
                f'  Directorio actual: {Path.cwd()}\n'
                'Configura IMAGES_DIR en la celda de configuración.'
            )
        return df
    base_dir = Path(base_dir)
    EXTS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
    idx  = {p.name: str(p) for p in base_dir.rglob('*') if p.suffix.lower() in EXTS}
    print(f'  {len(idx):,} imágenes indexadas en {base_dir}')
    df = df.copy()
    df['image_path'] = df['image_path'].map(lambda p: idx.get(Path(p).name, p))
    n_ok = df['image_path'].apply(lambda p: Path(p).exists()).sum()
    print(f'  ✔ {n_ok:,}/{len(df):,} rutas válidas tras remap')
    return df

# ── Carga del CSV ─────────────────────────────────────────────────────────────
if Path(CSV_BALANCED).exists():
    df = pd.read_csv(CSV_BALANCED, dtype={'nregistro': str})
    print(f'✔ Cargado: {CSV_BALANCED}')
elif Path(CSV_ORIGINAL).exists():
    df = pd.read_csv(CSV_ORIGINAL, dtype={'nregistro': str})
    df['augmented'] = False
    print(f'✔ Fallback: {CSV_ORIGINAL}')
else:
    raise FileNotFoundError(
        f'CSV no encontrado. Buscado en: {Path(CSV_BALANCED).resolve()}\n'
        f'Directorio actual: {Path.cwd()}'
    )

df = df[df['clase'].notna() & df['image_path'].notna()].copy().reset_index(drop=True)
df = remap_image_paths(df, IMAGES_DIR)

# ── Splits — solo imágenes ORIGINALES en train (aug se hace online) ───────────
df_train = df[(df['split']=='train') & (df['augmented']==False)].reset_index(drop=True)
df_val   = df[df['split']=='val'].reset_index(drop=True)
df_test  = df[df['split']=='test'].reset_index(drop=True)

# LabelEncoder
le = LabelEncoder()
le.fit(CLASES)

print(f'\nTrain (originales): {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')
print('\nDistribución clases train:')
for cls, n in df_train['clase'].value_counts().reindex(CLASES).items():
    bar = '█' * int(n / df_train['clase'].value_counts().max() * 25)
    print(f'  {cls:<40} {n:>5,}  {bar}')


---
# BLOQUE 1 — Dataset y DataLoaders

## 3. Transforms y clase MedDataset

### Augmentation online vs offline
En el notebook de ML el augmentation fue **offline** (imágenes precomputadas guardadas en disco).
En DL lo hacemos **online**: en cada época cada imagen recibe transformaciones aleatorias distintas.
Esto genera mucha más variedad (30 épocas × 1 imagen = 30 versiones distintas) sin coste de disco.

### ¿Por qué NO usamos las imágenes aug del CSV?
Las imágenes sintéticas del EDA no están disponibles en Lightning AI (solo se generaron en la máquina original). El augmentation online es superior: más variedad, cero coste de almacenamiento.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Train: augmentation activo
transform_train = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),            # resize directo — evita zonas grises
    T.RandomHorizontalFlip(p=0.3),             # suave: cajas tienen texto
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.25, contrast=0.20, saturation=0.20, hue=0.05),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val / Test: sin augmentation → evaluación determinista
transform_eval = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class MedDataset(Dataset):
    """Dataset de imágenes de medicamentos.
    Lanza error explícito si las rutas no son válidas (en lugar de silenciar con grises).
    """
    def __init__(self, df, transform, le):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.le        = le
        self.labels    = self.le.transform(self.df['clase'].values)

        if len(self.df) == 0:
            raise RuntimeError('MedDataset vacío — comprueba el CSV y el split.')
        # Diagnóstico rápido (sin filtrar)
        n_check = min(5, len(self.df))
        n_ok    = self.df['image_path'].head(n_check).apply(lambda p: Path(p).exists()).sum()
        if n_ok == 0:
            raise RuntimeError(
                f'Ninguna ruta válida en MedDataset.\n'
                f'  Ejemplo: {self.df["image_path"].iloc[0]}\n'
                f'  Configura IMAGES_DIR.'
            )

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(self.labels[idx])
        try:
            img = Image.open(row['image_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (200, 200, 200))
        return self.transform(img), label


ds_train = MedDataset(df_train, transform_train, le)
ds_val   = MedDataset(df_val,   transform_eval,  le)
ds_test  = MedDataset(df_test,  transform_eval,  le)

print(f'✔ Datasets — Train: {len(ds_train):,}  Val: {len(ds_val):,}  Test: {len(ds_test):,}')


## 4. DataLoaders con WeightedRandomSampler

El desbalance de clases se maneja con dos estrategias combinadas:
- **WeightedRandomSampler**: el DataLoader muestrea clases con igual probabilidad en cada batch.
- **Pesos en CrossEntropyLoss**: penaliza más los errores en clases minoritarias.

Esto es el equivalente en DL al `class_weight='balanced'` del ML.


In [ ]:
# Pesos para el sampler (inversamente proporcional al tamaño de clase)
lbl_train   = np.array(le.transform(df_train['clase'].values))
conteos_cls = np.bincount(lbl_train, minlength=N_CLASES)
pesos_cls   = 1.0 / (conteos_cls + 1e-6)
pesos_sample = torch.tensor(pesos_cls[lbl_train], dtype=torch.float)

sampler = WeightedRandomSampler(pesos_sample, len(pesos_sample), replacement=True)

# Pesos para CrossEntropyLoss
pesos_loss = torch.tensor(
    pesos_cls / pesos_cls.sum() * N_CLASES, dtype=torch.float
).to(DEVICE)

N_WORKERS = min(4, os.cpu_count() or 1)

dl_train = DataLoader(ds_train, batch_size=BATCH, sampler=sampler,
                      num_workers=N_WORKERS, pin_memory=(DEVICE.type=='cuda'), drop_last=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False,
                      num_workers=N_WORKERS, pin_memory=(DEVICE.type=='cuda'))
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False,
                      num_workers=N_WORKERS, pin_memory=(DEVICE.type=='cuda'))

print(f'DataLoaders — Train: {len(dl_train)} batches | Val: {len(dl_val)} | Test: {len(dl_test)}')
print(f'num_workers: {N_WORKERS}')
print('\nPesos por clase (CrossEntropyLoss):')
for i, cls in enumerate(CLASES):
    print(f'  {cls:<40}: {pesos_loss[i]:.4f}  (n_train={conteos_cls[i]:,})')


## 5. Visualización del batch

In [ ]:
def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (t * std + mean).clamp(0,1)

imgs_v, lbls_v = next(iter(dl_train))
imgs_v = imgs_v[:16]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, img, lbl in zip(axes.flat, imgs_v, lbls_v[:16]):
    ax.imshow(denormalize(img).permute(1,2,0).numpy())
    cls = le.classes_[lbl.item()]
    ax.set_title(cls[:20], fontsize=7, color=PALETTE[cls])
    ax.axis('off')
plt.suptitle('Batch de entrenamiento (16 imágenes) — con augmentation online activo',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dl_batch_sample.png', bbox_inches='tight')
plt.show()


---
# BLOQUE 2 — CNN From Scratch

## 6. Arquitectura MedCNN

### ¿Por qué empezar desde cero?

Antes de usar modelos preentrenados, diseñamos una CNN propia para entender:
1. Qué rendimiento se puede alcanzar **sin conocimiento previo** de ImageNet.
2. Qué problemas aparecen (overfitting, convergencia lenta) que el transfer learning resuelve.
3. Cuánto aporta cada mejora (dropout, scheduler, augmentation) de forma aislada.

### Arquitectura MedCNN

Cuatro bloques convolucionales con doble conv + BN + ReLU + MaxPool, inspirados en VGG:

```
Input (3×224×224)
  → Bloque 1: Conv(3→32) × 2 + MaxPool  → 32×112×112
  → Bloque 2: Conv(32→64) × 2 + MaxPool  → 64×56×56
  → Bloque 3: Conv(64→128) × 2 + MaxPool → 128×28×28
  → Bloque 4: Conv(128→256) + MaxPool    → 256×14×14
  → AdaptiveAvgPool → 256×1×1
  → Dropout → Linear(256→256) → ReLU → Dropout → Linear(256→5)
```

`AdaptiveAvgPool2d(1)` en lugar de `Flatten` directo permite usar la misma arquitectura
con cualquier tamaño de imagen de entrada — más robusta.


In [ ]:
class MedCNN(nn.Module):
    """CNN diseñada desde cero para clasificación multiclase de imágenes farmacéuticas.

    4 bloques Conv-BN-ReLU-MaxPool con número de canales creciente (32→64→128→256).
    Cabeza clasificadora con AdaptiveAvgPool + Dropout + Linear.
    """
    def __init__(self, n_classes=N_CLASES, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            # Bloque 1: 3 → 32 (224→112)
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Bloque 2: 32 → 64 (112→56)
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Bloque 3: 64 → 128 (56→28)
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Bloque 4: 128 → 256 (28→14)
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # → (B, 256, 1, 1)
            nn.Flatten(),              # → (B, 256)
            nn.Dropout(dropout),
            nn.Linear(256, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model_scratch = MedCNN(dropout=0.4).to(DEVICE)
n_params = sum(p.numel() for p in model_scratch.parameters() if p.requires_grad)
print(f'MedCNN — Parámetros entrenables: {n_params:,}')
print()
# Resumen de dimensiones
x_test = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    feat = model_scratch.features(x_test)
    print(f'Dimensiones tras features: {tuple(feat.shape)}')
    out = model_scratch(x_test)
    print(f'Dimensiones de salida:     {tuple(out.shape)}  (B × N_CLASES={N_CLASES})')


## 7. Funciones de entrenamiento y evaluación

Implementamos una función genérica que sirve tanto para CNN Scratch como para Transfer Learning.
Incluye early stopping sobre F1-macro (no accuracy) para manejar mejor el desbalance de clases.


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * len(labels)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    n   = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / n, acc, f1, np.array(all_preds), np.array(all_labels)


def train_model(model, model_name, criterion, optimizer, scheduler,
                n_epochs, patience=PATIENCE, unfreeze_ep=None):
    """Bucle completo de entrenamiento con early stopping sobre val F1-macro.
    Retorna el historial de métricas y restaura los mejores pesos automáticamente.
    """
    hist = {k: [] for k in ['tr_loss','val_loss','tr_acc','val_acc','val_f1','lr']}
    best_f1, best_w, sin_mejora = 0.0, None, 0
    back_desc = False

    print(f'\n{"═"*65}')
    print(f'  {model_name}')
    print(f'{"═"*65}')
    print(f'  {"Época":>5}  {"LR":>9}  {"Tr Loss":>8}  {"Tr Acc":>7}  '
          f'{"Val Loss":>8}  {"Val Acc":>7}  {"Val F1":>7}  Estado')
    print(f'  {"─"*63}')

    for ep in range(1, n_epochs + 1):
        # Fine-tuning progresivo: descongelar backbone
        if unfreeze_ep and ep == unfreeze_ep and not back_desc:
            for p in model.parameters(): p.requires_grad = True
            # Añadir parámetros del backbone al optimizer con lr baja
            back_params = [p for n, p in model.named_parameters()
                           if not any(x in n for x in ['classifier','fc']) and p.requires_grad]
            optimizer.add_param_group({'params': back_params, 'lr': LR_TL_BACK})
            back_desc = True
            print(f'  [Época {ep}] ✔ Backbone descongelado (lr={LR_TL_BACK})')

        tr_loss, tr_acc   = train_epoch(model, dl_train, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_f1, _, _ = eval_epoch(model, dl_val, criterion, DEVICE)
        lr_actual = optimizer.param_groups[0]['lr']

        if hasattr(scheduler, 'step'):
            try:    scheduler.step(val_loss)
            except: scheduler.step()

        hist['tr_loss'].append(tr_loss);  hist['val_loss'].append(val_loss)
        hist['tr_acc'].append(tr_acc);    hist['val_acc'].append(val_acc)
        hist['val_f1'].append(val_f1);    hist['lr'].append(lr_actual)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_w  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            sin_mejora = 0; estado = '★ mejor'
        else:
            sin_mejora += 1
            estado = f'({sin_mejora}/{patience})'

        if ep == 1 or ep % 5 == 0 or sin_mejora == 0 or sin_mejora >= patience:
            print(f'  {ep:>5}  {lr_actual:>9.2e}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  '
                  f'{val_loss:>8.4f}  {val_acc:>7.4f}  {val_f1:>7.4f}  {estado}')

        if sin_mejora >= patience:
            print(f'  Early stopping en época {ep}'); break

    model.load_state_dict(best_w); model.to(DEVICE)
    print(f'\n  ✔ Mejor Val F1-macro: {best_f1:.4f}')
    return hist, best_f1


def plot_curves(hist, title, save_name):
    """Curvas de accuracy y loss train vs val con diagnóstico de overfitting."""
    epocas = range(1, len(hist['tr_loss']) + 1)
    gap_acc  = max(hist['tr_acc'])  - max(hist['val_acc'])
    gap_loss = hist['val_loss'][-1] - hist['tr_loss'][-1]

    if gap_acc > 0.10:   overfit_txt = f'⚠ Overfitting (gap acc={gap_acc:.3f})'
    elif gap_acc > 0.05: overfit_txt = f'⚠ Overfitting leve (gap={gap_acc:.3f})'
    else:                overfit_txt = f'✔ Sin overfitting significativo (gap={gap_acc:.3f})'

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Accuracy
    axes[0].plot(epocas, hist['tr_acc'],  label='Train', color='#4C72B0', lw=2)
    axes[0].plot(epocas, hist['val_acc'], label='Val',   color='#DD8452', lw=2, ls='--')
    axes[0].axhline(max(hist['val_acc']), color='green', ls=':', lw=1,
                    label=f'Best val={max(hist["val_acc"]):.3f}')
    axes[0].set_title(f'Accuracy\n{overfit_txt}', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.05)

    # Loss
    axes[1].plot(epocas, hist['tr_loss'],  label='Train', color='#4C72B0', lw=2)
    axes[1].plot(epocas, hist['val_loss'], label='Val',   color='#DD8452', lw=2, ls='--')
    axes[1].set_title('Loss (CrossEntropy)', fontweight='bold')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=8)

    # Val F1-macro
    axes[2].plot(epocas, hist['val_f1'], color='#55A868', lw=2, label='Val F1-macro')
    axes[2].axhline(max(hist['val_f1']), color='#55A868', ls=':', lw=1,
                    label=f'Best={max(hist["val_f1"]):.3f}')
    axes[2].set_title('Val F1-macro', fontweight='bold')
    axes[2].set_xlabel('Época'); axes[2].set_ylabel('F1-macro')
    axes[2].legend(fontsize=8)

    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'dl_curves_{save_name}.png', bbox_inches='tight')
    plt.show()

    print(f'  Train acc final: {hist["tr_acc"][-1]:.4f}')
    print(f'  Val acc final:   {hist["val_acc"][-1]:.4f}')
    print(f'  Gap acc:         {gap_acc:+.4f}  |  Gap loss: {gap_loss:+.4f}')


## 8. Entrenamiento CNN Scratch — Versión base (V1)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=pesos_loss, label_smoothing=0.05)

optimizer_s1 = optim.Adam(model_scratch.parameters(), lr=LR_SCRATCH, weight_decay=1e-4)
scheduler_s1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s1, mode='min', patience=3, factor=0.5)

hist_s1, f1_s1 = train_model(
    model_scratch, 'CNN From Scratch — V1 (base)',
    criterion, optimizer_s1, scheduler_s1,
    n_epochs=N_EPOCHS_SCRATCH
)
plot_curves(hist_s1, 'CNN From Scratch — V1 (base)', 'scratch_v1')


## 9. Entrenamiento CNN Scratch — Versión mejorada (V2)

Misma arquitectura, pero con tres cambios para reducir el overfitting visto en V1:
- **Más dropout** (0.5 vs 0.4): penaliza más la dependencia entre neuronas.
- **CosineAnnealingLR**: el learning rate baja suavemente hasta `eta_min=1e-6`, lo que permite afinar el modelo al final sin que los gradientes destruyan lo aprendido.
- **Weight decay mayor** (2e-4 vs 1e-4): regularización L2 adicional sobre los pesos.


In [ ]:
model_scratch_v2 = MedCNN(dropout=0.5).to(DEVICE)

optimizer_s2 = optim.AdamW(model_scratch_v2.parameters(), lr=LR_SCRATCH, weight_decay=2e-4)
scheduler_s2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s2, T_max=N_EPOCHS_SCRATCH, eta_min=1e-6)

hist_s2, f1_s2 = train_model(
    model_scratch_v2, 'CNN From Scratch — V2 (mejorada)',
    criterion, optimizer_s2, scheduler_s2,
    n_epochs=N_EPOCHS_SCRATCH
)
plot_curves(hist_s2, 'CNN From Scratch — V2 (mejorada: más dropout + cosine LR)', 'scratch_v2')

print('\n── Comparativa V1 vs V2 ──')
print(f'  V1 (base):     Best Val F1 = {f1_s1:.4f}')
print(f'  V2 (mejorada): Best Val F1 = {f1_s2:.4f}')
print(f'  Δ V2 − V1:     {f1_s2 - f1_s1:+.4f}')
if f1_s2 > f1_s1:
    print('  → Las mejoras (dropout + cosine LR) reducen overfitting y mejoran generalización.')
else:
    print('  → Mejora marginal: el cuello de botella es la falta de representaciones preentrenadas.')


---
# BLOQUE 3 — Transfer Learning

## ¿Por qué Transfer Learning supera a CNN From Scratch?

ResNet-50 y MobileNetV2 fueron preentrenadas en **ImageNet** (1.3M imágenes, 1000 clases).
Sus primeras capas detectan bordes, texturas y formas genéricas. Las últimas capas detectan
objetos complejos. Todo ese conocimiento se transfiere a nuestro problema de medicamentos.

Con un dataset de ~7.000 imágenes (solo originales), entrenar desde cero produce overfitting
inevitable — no hay suficientes datos para aprender buenas representaciones visuales. Con
transfer learning partimos de representaciones ya buenas y solo necesitamos ajustar la cabeza.

### Estrategia de fine-tuning progresivo

| Fase | Épocas | Qué se entrena | LR |
|------|--------|----------------|----|
| 1 — Feature Extractor | 1 a `UNFREEZE_EPOCH` | Solo cabeza nueva | `LR_TL_HEAD=3e-4` |
| 2 — Fine-Tuning | `UNFREEZE_EPOCH` en adelante | Todo el modelo | backbone: `LR_TL_BACK=3e-5` |

La lr del backbone es 10× menor para no destruir las representaciones ImageNet aprendidas.

## 10. MobileNetV2 — Feature Extractor (backbone congelado)


In [ ]:
def build_mobilenet(n_classes=N_CLASES, dropout=0.4, freeze=True):
    """MobileNetV2 con cabeza personalizada para N_CLASES.
    Si freeze=True, backbone completamente congelado (Feature Extractor).
    """
    m = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    if freeze:
        for p in m.parameters(): p.requires_grad = False
    in_features = m.last_channel  # 1280
    m.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout / 2),
        nn.Linear(512, n_classes),
    )
    for p in m.classifier.parameters(): p.requires_grad = True
    return m


model_mn_feat = build_mobilenet(freeze=True).to(DEVICE)
n_trainable   = sum(p.numel() for p in model_mn_feat.parameters() if p.requires_grad)
n_total       = sum(p.numel() for p in model_mn_feat.parameters())
print(f'MobileNetV2 Feature Extractor')
print(f'  Parámetros totales:      {n_total:,}')
print(f'  Parámetros entrenables:  {n_trainable:,}  ({n_trainable/n_total*100:.1f}%)')

optimizer_mn_feat = optim.Adam(
    filter(lambda p: p.requires_grad, model_mn_feat.parameters()),
    lr=LR_TL_HEAD, weight_decay=1e-4
)
scheduler_mn_feat = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_mn_feat, T_max=N_EPOCHS_TL, eta_min=1e-6)

hist_mn_feat, f1_mn_feat = train_model(
    model_mn_feat, 'MobileNetV2 — Feature Extractor (backbone congelado)',
    criterion, optimizer_mn_feat, scheduler_mn_feat,
    n_epochs=N_EPOCHS_TL
)
plot_curves(hist_mn_feat, 'MobileNetV2 — Feature Extractor', 'mn_feat')


## 11. MobileNetV2 — Fine-Tuning progresivo

In [ ]:
model_mn_ft = build_mobilenet(freeze=True).to(DEVICE)

optimizer_mn_ft = optim.AdamW(
    filter(lambda p: p.requires_grad, model_mn_ft.parameters()),
    lr=LR_TL_HEAD, weight_decay=1e-4
)
scheduler_mn_ft = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_mn_ft, T_max=N_EPOCHS_TL, eta_min=1e-6)

hist_mn_ft, f1_mn_ft = train_model(
    model_mn_ft, 'MobileNetV2 — Fine-Tuning progresivo',
    criterion, optimizer_mn_ft, scheduler_mn_ft,
    n_epochs=N_EPOCHS_TL, unfreeze_ep=UNFREEZE_EPOCH
)
plot_curves(hist_mn_ft, 'MobileNetV2 — Fine-Tuning progresivo', 'mn_ft')

print('\n── Feature Extractor vs Fine-Tuning ──')
print(f'  MobileNetV2 Feat. Extractor: Val F1 = {f1_mn_feat:.4f}')
print(f'  MobileNetV2 Fine-Tuning:     Val F1 = {f1_mn_ft:.4f}')
print(f'  Δ FT − FE: {f1_mn_ft - f1_mn_feat:+.4f}')


## 12. ResNet-50 — Fine-Tuning

In [ ]:
def build_resnet50(n_classes=N_CLASES, dropout=0.4):
    """ResNet-50 preentrenado con cabeza personalizada. Backbone inicialmente congelado."""    m = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    for p in m.parameters(): p.requires_grad = False
    in_features = m.fc.in_features  # 2048
    m.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(512, n_classes),
    )
    for p in m.fc.parameters(): p.requires_grad = True
    return m


model_rn = build_resnet50().to(DEVICE)
n_trainable_rn = sum(p.numel() for p in model_rn.parameters() if p.requires_grad)
n_total_rn     = sum(p.numel() for p in model_rn.parameters())
print(f'ResNet-50')
print(f'  Parámetros totales:      {n_total_rn:,}')
print(f'  Parámetros entrenables:  {n_trainable_rn:,}  ({n_trainable_rn/n_total_rn*100:.1f}%)')

optimizer_rn = optim.AdamW(
    filter(lambda p: p.requires_grad, model_rn.parameters()),
    lr=LR_TL_HEAD, weight_decay=1e-4
)
scheduler_rn = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_rn, T_max=N_EPOCHS_TL, eta_min=1e-6)

hist_rn, f1_rn = train_model(
    model_rn, 'ResNet-50 — Fine-Tuning progresivo',
    criterion, optimizer_rn, scheduler_rn,
    n_epochs=N_EPOCHS_TL, unfreeze_ep=UNFREEZE_EPOCH
)
plot_curves(hist_rn, 'ResNet-50 — Fine-Tuning progresivo', 'rn50_ft')


---
# BLOQUE 4 — Evaluación final

## 13. Evaluación en test de todos los modelos

El test se toca **una única vez** por modelo. Hemos tomado todas las decisiones mirando la validación.


In [ ]:
todos_modelos = {
    'CNN Scratch V1':            model_scratch,
    'CNN Scratch V2':            model_scratch_v2,
    'MobileNetV2 Feat.Ext.':    model_mn_feat,
    'MobileNetV2 Fine-Tuning':  model_mn_ft,
    'ResNet-50 Fine-Tuning':    model_rn,
}

test_results = {}

print('⚠️  EVALUACIÓN FINAL EN TEST — TODOS LOS MODELOS')
print('='*70)
print(f'  {"Modelo":<35} {"Acc Test":>9} {"F1 Test":>9}')
print('─'*70)

for nombre, modelo in todos_modelos.items():
    _, acc_t, f1_t, preds_t, true_t = eval_epoch(modelo, dl_test, criterion, DEVICE)
    test_results[nombre] = {
        'acc': acc_t, 'f1': f1_t,
        'preds': preds_t, 'true': true_t,
        'cm': confusion_matrix(true_t, preds_t),
    }
    print(f'  {nombre:<35} {acc_t:>9.4f} {f1_t:>9.4f}')

print('='*70)
mejor_nombre = max(test_results, key=lambda k: test_results[k]['f1'])
print(f'\n  Mejor modelo: {mejor_nombre}')
print(f'  Acc={test_results[mejor_nombre]["acc"]:.4f}  F1={test_results[mejor_nombre]["f1"]:.4f}')


## 14. Matrices de confusión — Test

In [ ]:
n_modelos = len(todos_modelos)
fig, axes = plt.subplots(1, n_modelos, figsize=(4.5 * n_modelos, 4.5))

for ax, (nombre, res) in zip(axes, test_results.items()):
    cm_norm = res['cm'].astype(float) / (res['cm'].sum(axis=1, keepdims=True) + 1e-9)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=[c[:8] for c in CLASES],
                yticklabels=[c[:8] for c in CLASES],
                cbar=False, ax=ax, annot_kws={'size': 7})
    ax.set_title(f'{nombre}\nAcc={res["acc"]:.3f}  F1={res["f1"]:.3f}',
                 fontsize=8, fontweight='bold')
    ax.set_xlabel('Predicción', fontsize=7)
    ax.set_ylabel('Real', fontsize=7)
    ax.tick_params(axis='both', labelsize=6)

plt.suptitle('Matrices de Confusión — Test (normalizada por fila)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dl_confusion_matrices.png', bbox_inches='tight')
plt.show()


## 15. Análisis de errores — imágenes mal clasificadas

In [ ]:
mejor_modelo = todos_modelos[mejor_nombre]
_, _, _, preds_val, true_val = eval_epoch(mejor_modelo, dl_val, criterion, DEVICE)

df_val_eval = df_val.copy().reset_index(drop=True)
df_val_eval = df_val_eval.iloc[:len(preds_val)].copy()
df_val_eval['y_real'] = le.inverse_transform(true_val)
df_val_eval['y_pred'] = le.inverse_transform(preds_val)
df_val_eval['correcto'] = df_val_eval['y_real'] == df_val_eval['y_pred']

print(f'Accuracy por tipo de imagen ({mejor_nombre}):')
print(df_val_eval.groupby('image_type')['correcto']
      .agg(['mean','count']).round(4).rename(columns={'mean':'accuracy'}).to_string())

pares = (df_val_eval[~df_val_eval['correcto']]
         .groupby(['y_real','y_pred']).size()
         .sort_values(ascending=False).head(6))
print('\nTop 6 pares de confusión más frecuentes (val):')
print(pares.to_string())

top3 = pares.head(3).index.tolist()
if top3:
    fig, axes = plt.subplots(len(top3), 6, figsize=(16, len(top3)*3.2))
    if len(top3) == 1: axes = [axes]
    for row, (cls_r, cls_p) in enumerate(top3):
        ejs = df_val_eval[(df_val_eval['y_real']==cls_r) & (df_val_eval['y_pred']==cls_p)].head(6)
        for col in range(6):
            ax = axes[row][col]
            if col < len(ejs):
                try: ax.imshow(Image.open(ejs.iloc[col]['image_path']).convert('RGB'))
                except: ax.set_facecolor('#ddd')
            else: ax.set_facecolor('#f0f0f0')
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(f'Real: {cls_r[:14]}\nPred: {cls_p[:14]}',
                              fontsize=7, color='#C44E52', rotation=0, ha='right')
    plt.suptitle(f'Errores más frecuentes — {mejor_nombre} (val)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'dl_errores_visualizados.png', bbox_inches='tight')
    plt.show()


## 16. Grad-CAM — ¿qué zonas activa la red?

**Grad-CAM** superpone un mapa de calor sobre la imagen mostrando qué regiones fueron
decisivas para la predicción. Si el modelo mira el envase → aprende lo correcto.
Si mira el fondo blanco → hay sesgo de dataset.


In [ ]:
class GradCAM:
    """Implementación minimal de Grad-CAM para ResNet-50 y MobileNetV2."""    def __init__(self, model, target_layer):
        self.model        = model
        self.activations  = None
        self.gradients    = None
        self._hooks = [
            target_layer.register_forward_hook(
                lambda m, i, o: setattr(self, 'activations', o.detach())),
            target_layer.register_full_backward_hook(
                lambda m, gi, go: setattr(self, 'gradients', go[0].detach())),
        ]

    def __call__(self, img_t, class_idx=None):
        self.model.eval()
        logits = self.model(img_t)
        if class_idx is None: class_idx = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam     = torch.clamp((weights * self.activations).sum(1).squeeze(), min=0)
        cam     = cam / (cam.max() + 1e-8)
        cam_np  = cv2.resize(cam.cpu().numpy(), (IMG_SIZE, IMG_SIZE))
        return cam_np, class_idx

    def remove(self):
        for h in self._hooks: h.remove()


def overlay_cam(img_pil, cam_np):
    img_np  = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    heatmap = cv2.applyColorMap(np.uint8(255*cam_np), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return np.uint8(0.45 * heatmap + 0.55 * img_np)


# Grad-CAM sobre ResNet-50 (layer4[-1].conv3)
try:
    gcam = GradCAM(model_rn, model_rn.layer4[-1].conv3)
    fig, axes = plt.subplots(2, N_CLASES, figsize=(N_CLASES*3.5, 7))
    for col, cls in enumerate(CLASES):
        sub = df_val_eval[(df_val_eval['y_real']==cls) & df_val_eval['correcto']].head(1)
        if len(sub) == 0:
            axes[0,col].axis('off'); axes[1,col].axis('off'); continue
        try:
            img_pil = Image.open(sub.iloc[0]['image_path']).convert('RGB')
            img_t   = transform_eval(img_pil).unsqueeze(0).to(DEVICE)
            cam_np, pred_idx = gcam(img_t)
            axes[0,col].imshow(img_pil.resize((IMG_SIZE, IMG_SIZE)))
            axes[0,col].set_title(cls[:18], fontsize=8, color=PALETTE[cls], fontweight='bold')
            axes[1,col].imshow(overlay_cam(img_pil, cam_np))
            axes[1,col].set_title(f'Pred: {le.classes_[pred_idx][:14]}', fontsize=7)
        except Exception as e:
            axes[0,col].axis('off'); axes[1,col].axis('off')
        axes[0,col].axis('off'); axes[1,col].axis('off')
    axes[0,0].set_ylabel('Original', fontsize=9)
    axes[1,0].set_ylabel('Grad-CAM',  fontsize=9)
    plt.suptitle('Grad-CAM — ResNet-50 (imágenes correctamente clasificadas de val)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'dl_gradcam.png', bbox_inches='tight')
    plt.show()
    gcam.remove()
except Exception as e:
    print(f'Grad-CAM no disponible: {e}')


---
# BLOQUE 5 — Comparativa global

## 17. DL vs ML clásico — comparativa final


In [ ]:
# Cargar resultados del ML si existen
ml_csv = Path('output_ml') / 'comparativa_modelos.csv'
ml_ref = {}
if ml_csv.exists():
    df_ml = pd.read_csv(ml_csv)
    mejor_ml = df_ml.sort_values('Accuracy', ascending=False).iloc[0]
    ml_ref[mejor_ml['Modelo']] = {
        'acc': mejor_ml['Accuracy'],
        'f1':  mejor_ml.get('F1-macro', mejor_ml.get('F1_macro', float('nan'))),
        'tipo': 'ML clásico'
    }
    print(f'ML mejor modelo: {mejor_ml["Modelo"]}  Acc={mejor_ml["Accuracy"]:.4f}')
else:
    print('⚠ No se encontró output_ml/comparativa_modelos.csv')
    print('  Introduce manualmente los resultados del ML en ml_ref:')
    # ml_ref['SVM RBF (mejor ML)'] = {'acc': 0.63, 'f1': 0.59, 'tipo': 'ML clásico'}

# Combinar resultados
todos_res = {}
for nombre, datos in ml_ref.items():
    todos_res[nombre] = datos
for nombre, res in test_results.items():
    tipo = 'CNN Scratch' if 'Scratch' in nombre else 'Transfer Learning'
    todos_res[nombre] = {'acc': res['acc'], 'f1': res['f1'], 'tipo': tipo}

df_comp = pd.DataFrame(todos_res).T.reset_index().rename(columns={'index':'Modelo'})
df_comp[['acc','f1']] = df_comp[['acc','f1']].astype(float)
df_comp = df_comp.sort_values('f1', ascending=False).reset_index(drop=True)

print('\n' + '='*65)
print('  COMPARATIVA GLOBAL — Accuracy y F1-macro en Test')
print('='*65)
print(df_comp[['Modelo','tipo','acc','f1']].round(4).to_string(index=False))
print('='*65)

# Gráfico comparativo
colores_tipo = {'ML clásico': '#8172B2', 'CNN Scratch': '#DD8452', 'Transfer Learning': '#4C72B0'}
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, metrica, titulo in [(axes[0], 'acc', 'Accuracy en Test'), (axes[1], 'f1', 'F1-macro en Test')]:
    colores_bar = [colores_tipo.get(t, '#888') for t in df_comp['tipo']]
    bars = ax.barh(df_comp['Modelo'], df_comp[metrica], color=colores_bar, edgecolor='white', height=0.6)
    for bar, v in zip(bars, df_comp[metrica]):
        ax.text(v + 0.003, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=8, fontweight='bold')
    ax.set_xlim(0, min(1.05, df_comp[metrica].max() + 0.08))
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel(titulo.split(' ')[0])

# Leyenda de tipos
from matplotlib.patches import Patch
leyenda = [Patch(facecolor=c, label=t) for t, c in colores_tipo.items()]
fig.legend(handles=leyenda, loc='lower right', ncol=3, fontsize=9)

plt.suptitle('Comparativa Global: ML Clásico → CNN Scratch → Transfer Learning',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dl_comparativa_global.png', bbox_inches='tight')
plt.show()


## 18. Conclusiones

In [ ]:
# Tabla resumen para guardar
df_comp.to_csv(OUTPUT_DIR / 'comparativa_dl_completa.csv', index=False)

# Guardar mejor modelo
torch.save(todos_modelos[mejor_nombre].state_dict(),
           OUTPUT_DIR / f'mejor_modelo_{mejor_nombre.replace(" ","_")}.pt')
joblib.dump(le, OUTPUT_DIR / 'label_encoder_dl.pkl')

linea = '═'*72
print(linea)
print('  RESUMEN EJECUTIVO — Deep Learning')
print(linea)
print()
print(f'  {"Modelo":<38} {"Acc":>7} {"F1":>7}  Tipo')
print('─'*72)
for _, row in df_comp.iterrows():
    marca = ' ★' if row['Modelo'] == mejor_nombre else ''
    print(f'  {row["Modelo"]:<38} {row["acc"]:>7.4f} {row["f1"]:>7.4f}  {row["tipo"]}{marca}')
print(linea)
print()

# Δ DL vs ML
if ml_ref:
    mejor_ml_acc = max(v['acc'] for v in ml_ref.values())
    mejor_ml_f1  = max(v['f1']  for v in ml_ref.values())
    delta_acc = test_results[mejor_nombre]['acc'] - mejor_ml_acc
    delta_f1  = test_results[mejor_nombre]['f1']  - mejor_ml_f1
    print(f'  Δ mejor DL − mejor ML:  Acc={delta_acc:+.4f}  F1={delta_f1:+.4f}')
    if delta_acc > 0.05:
        print('  → El DL supera claramente al ML. Transfer learning compensa la falta de features manuales.')
    elif delta_acc > 0:
        print('  → El DL supera moderadamente al ML.')
    else:
        print('  → El ML es competitivo. El cuello de botella es el tamaño del dataset, no el modelo.')
    print()

print('  ANÁLISIS DE OVERFITTING (últimas épocas):')
for nombre, hist in [('CNN Scratch V1', hist_s1), ('CNN Scratch V2', hist_s2),
                      ('MobileNetV2 FE', hist_mn_feat), ('MobileNetV2 FT', hist_mn_ft),
                      ('ResNet-50 FT',   hist_rn)]:
    gap = max(hist['tr_acc']) - max(hist['val_acc'])
    estado = '⚠ overfit' if gap > 0.08 else ('⚠ leve' if gap > 0.04 else '✔ OK')
    print(f'  {nombre:<30} gap acc = {gap:+.3f}  {estado}')
print()
print('  Archivos generados:')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'    · {f.name}')
print(linea)


### Interpretación de resultados

**Progresión esperada de rendimiento:**

| Modelo | Razón del cambio | Δ esperado |
|--------|-----------------|-----------|
| CNN Scratch V1 → V2 | Más dropout + cosine LR → menos overfitting | +1-3 pp F1 |
| Scratch → MobileNetV2 FE | Representaciones ImageNet preentrenadas | +5-10 pp F1 |
| FE → Fine-Tuning | Backbone se adapta al dominio farmacéutico | +2-5 pp F1 |
| MobileNetV2 FT → ResNet-50 FT | Backbone más potente (23M vs 3.4M params) | +1-3 pp F1 |

**CNN Scratch — diagnóstico de overfitting**: las CNN entrenadas desde cero con ~7k imágenes casi siempre muestran overfitting (train_acc >> val_acc). La mejora de V1 a V2 reduce este gap pero no lo elimina porque el modelo no tiene suficientes datos para aprender representaciones robustas desde cero.

**Transfer Learning Feature Extractor**: el backbone congelado actúa como un extractor de features fijo. Con tan solo entrenar la cabeza (~500K params) se supera a la CNN Scratch completa (~1M params). Esto demuestra que las representaciones ImageNet son transferibles incluso a imágenes de medicamentos.

**Fine-Tuning progresivo**: descongelar el backbone con lr muy baja (3e-5) permite que las últimas capas del backbone aprendan patterns específicos del packaging farmacéutico (tipografías, colores de envase, formas de blíster) sin destruir las representaciones genéricas de las primeras capas.

**Grad-CAM**: si los mapas de activación se concentran sobre el envase/pastilla y no sobre el fondo blanco, el modelo está aprendiendo señales correctas. Si mira el fondo, hay sesgo de dataset que se puede corregir con segmentación de fondo o augmentation más agresivo.

**Clase "otros"**: sigue siendo la más difícil en DL, igual que en ML. La razón es estructural: agrupa categorías terapéuticas heterogéneas sin patrón visual común. Un sistema de dos etapas (detector `otros` + clasificador 4 clases) podría mejorar esto.
